In [ ]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

In [ ]:
from datetime import datetime, timedelta
import random

import polars as pl

from corr_vars_widget import (
    ObsWidget,
    ObsmWidget,
    JsonWidget,
    JsonmWidget,
    TimeseriesWidget
)

In [ ]:
df = pl.DataFrame({"a": [1, 2, 3] * 10_000, "b": ["This", "is a", "test"] * 10_000})
creation_time = datetime.now()
obs = ObsWidget(df, "ICU Stay", creation_time=creation_time)
obs

In [ ]:
print(obs)

In [ ]:
df = pl.DataFrame({"a": [1, 2, 3] * 10_000, "b": ["This", "is a", "test"] * 10_000})
obsm = ObsmWidget({"df1": df, "df2": df.with_columns(pl.col("a") * 2)})
obsm

In [ ]:
print(obsm)

In [ ]:
obsm.sql

In [ ]:
obsm.data

In [ ]:
test_config = {
    "a": [1, 2, 3],
    "b": ["This", "is a", "test"],
    "bool": True,
    "null": None,
    "nested": {
        "c": [4, 5, 6],
        "d": {"e": "Nested value"},
        "empty_list": [],
        "empty_dict": {},
    },
}

In [ ]:
var_config = {
    "icu_length_of_stay": {
        "type": "derived_static",
        "compatible_with": ["icu_stay"],
        "requires": ["icu_admission", "icu_discharge"],
        "expression": "IF(icu_discharge - icu_admission IS NULL, NULL, GREATEST((EXTRACT(EPOCH FROM icu_discharge) - EXTRACT(EPOCH FROM icu_admission)) / (24 * 60 * 60), 1.0))",
    },
    "hospital_length_of_stay": {
        "type": "derived_static",
        "requires": ["hospital_admission", "hospital_discharge"],
        "expression": "IF(hospital_discharge - hospital_admission IS NULL, NULL, GREATEST((EXTRACT(EPOCH FROM hospital_discharge) - EXTRACT(EPOCH FROM hospital_admission)) / (24 * 60 * 60), 1.0))",
    },
    "campus_id": {
        "type": "derived_static",
        "compatible_with": ["icu_stay"],
        "requires": ["icu_id"],
        "expression": "SUBSTR(ARRAY_GET(icu_id, 1), 1, 1)",
    },
    "blood_sodium": {
        "type": "native_dynamic",
        "table": "it_ishmed_labor",
        "where": "(c_katalog_leistungtext IN ('Natrium HP', 'BGA-Natrium', 'Natrium (POC)', 'NATRIUM')) OR (c_katalog_leistungtext LIKE 'NATRIUM(BG)%') OR (c_katalog_leistungtext='Natrium' AND c_leistungnr IN ('N_101000', 'N_303011', 'C_NA', 'N_204100', 'N_701000'))",
        "value_dtype": "DOUBLE",
        "cleaning": {"value": {"low": 80, "high": 190}},
    },
    "blood_hematocrit": {
        "type": "native_dynamic",
        "table": "it_ishmed_labor",
        "where": "c_katalog_leistungtext LIKE '%Hämatokr%' AND NOT (c_leistungnr = 'H_HKTI' OR c_leistungnr = 'N_204140')",
        "value_dtype": "FLOAT",
        "cleaning": {"value": {"low": 0.0, "high": 1.0}},
    },
}

In [ ]:
json_widget = JsonWidget(test_config)
json_widget

In [ ]:
jsonm = JsonmWidget({"cub_hdp": var_config, "test": test_config})
jsonm

In [ ]:
t0 = datetime(2024, 2, 12, 7, 0)
random.seed(0)


def intervals(stay_id, labels, step_hours):
    """One row per label, laid end to end."""
    rows = []
    t = t0
    for label in labels:
        end = t + timedelta(hours=step_hours)
        rows.append({"stay_id": stay_id, "start": t, "end": end, "value": label})
        t = end
    return rows


def measurements(stay_id, n, lo, hi):
    return [
        {
            "stay_id": stay_id,
            "start": t0 + timedelta(minutes=15 * i),
            "value": random.uniform(lo, hi),
        }
        for i in range(n)
    ]


devices = pl.DataFrame(
    intervals(1, ["Ventilation anesthesia", "Ventilation adult"], 4)
    + intervals(2, ["Ventilation adult"], 6)
)
modes = pl.DataFrame(
    intervals(1, ["MAN/SPONT", "PCV", "BIPAP", "CPAP"], 2)
    + intervals(2, ["PCV", "CPAP"], 3)
)
crs = pl.DataFrame(measurements(1, 32, 40, 60) + measurements(2, 24, 35, 55))
fio2 = pl.DataFrame(measurements(1, 32, 0.3, 1.0) + measurements(2, 24, 0.2, 0.9))

In [ ]:
ts = TimeseriesWidget(
    intervals={"Device (name)": devices, "Ventilator mode": modes},
    values={"CRS": crs, "FiO2": fio2},
    id_col="stay_id",
    start_col="start",
    end_col="end",
    value_col="value",
)
ts

In [ ]:
print(ts)

In [ ]:
ts.data